# Week 5 Interim Data Profile

I used the provided FraudShield case files to do an initial profile before deciding what the data can support. The goal was to check structure, missing fields, and early feasibility for a reviewer-facing fraud workflow.


## Files Reviewed

I reviewed the evidence summary for account AC-1589269 and the wallet import sample for account AC-4471021. The evidence file gives case-level risk context. The wallet file gives transaction-level fields that can be checked for missingness and basic consistency.


In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

data_dir = Path('../data')
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)


In [2]:
evidence = pd.read_csv(data_dir / 'evidence_summary_ac_1589269.csv')
wallet = pd.read_csv(data_dir / 'wallet_import_ac_4471021.csv')

wallet['timestamp'] = pd.to_datetime(wallet['timestamp'], errors='coerce')
wallet['amount'] = pd.to_numeric(wallet['amount'], errors='coerce')

print('Evidence summary rows/columns:', evidence.shape)
print('Wallet import rows/columns:', wallet.shape)


Evidence summary rows/columns: (6, 6)
Wallet import rows/columns: (5, 9)


In [3]:
print('Evidence summary column types:')
print(evidence.dtypes)
print('\nWallet import column types after basic conversion:')
print(wallet.dtypes)


Evidence summary column types:
evidence_id    object
source         object
detail         object
use            object
account_id     object
risk_score      int64

Wallet import column types after basic conversion:
id                                      object
account_id                              object
timestamp            datetime64[ns, UTC-04:00]
type                                    object
amount                                   int64
payment_brand                           object
extra_details                           object
card_bin                               float64
chargeback_linked                         bool


In [4]:
missing_rows = []
for name, df in [('Evidence summary', evidence), ('Wallet import', wallet)]:
    missing = df.isna().sum()
    for column, count in missing.items():
        missing_rows.append({
            'file': name,
            'column': column,
            'missing_count': int(count),
            'missing_percent': round((count / len(df)) * 100, 1) if len(df) else 0,
        })

missing_df = pd.DataFrame(missing_rows)
print(missing_df.to_string(index=False))


            file            column  missing_count  missing_percent
Evidence summary       evidence_id              0              0.0
Evidence summary            source              0              0.0
Evidence summary            detail              0              0.0
Evidence summary               use              0              0.0
Evidence summary        account_id              0              0.0
Evidence summary        risk_score              0              0.0
   Wallet import                id              0              0.0
   Wallet import        account_id              0              0.0
   Wallet import         timestamp              0              0.0
   Wallet import              type              0              0.0
   Wallet import            amount              0              0.0
   Wallet import     payment_brand              0              0.0
   Wallet import     extra_details              0              0.0
   Wallet import          card_bin              2             

In [5]:
plot_df = missing_df.copy()
plot_df['label'] = plot_df['file'] + ' | ' + plot_df['column']
plot_df = plot_df.sort_values(['missing_count', 'label'], ascending=[True, True])

fig_height = max(5, len(plot_df) * 0.32)
fig, ax = plt.subplots(figsize=(10, fig_height))
colors = ['#6b6b6b' if value == 0 else '#a15c38' for value in plot_df['missing_count']]
ax.barh(plot_df['label'], plot_df['missing_count'], color=colors)
ax.set_title('Missing Values by File and Column')
ax.set_xlabel('Missing values')
ax.set_ylabel('')
ax.grid(axis='x', alpha=0.25)
for i, row in enumerate(plot_df.itertuples(index=False)):
    ax.text(row.missing_count + 0.03, i, f'{row.missing_count} ({row.missing_percent}%)', va='center', fontsize=8)
ax.set_xlim(0, max(2, plot_df['missing_count'].max() + 1))
plt.tight_layout()
fig.savefig(output_dir / 'missingness_summary.png', dpi=180)
fig.savefig(output_dir / 'missingness_summary.svg')
print('Saved missingness plot to ../outputs/missingness_summary.png and ../outputs/missingness_summary.svg')


Saved missingness plot to ../outputs/missingness_summary.png and ../outputs/missingness_summary.svg


In [6]:
print('Wallet transaction types:')
print(wallet['type'].value_counts())
print('\nAmount summary:')
print(wallet['amount'].describe())
print('\nMissing card_bin by transaction type:')
print(wallet.groupby('type')['card_bin'].apply(lambda s: int(s.isna().sum())))

card_deposits = wallet[(wallet['type'].eq('DEPOSIT')) & (wallet['payment_brand'].isin(['Visa', 'Mastercard']))]
print('\nCard deposits missing card_bin:', int(card_deposits['card_bin'].isna().sum()))


Wallet transaction types:
type
DEPOSIT       3
WITHDRAWAL    2

Amount summary:
count       5.000000
mean     3680.000000
std      2193.342199
min       950.000000
25%      1800.000000
50%      4750.000000
75%      4800.000000
max      6100.000000

Missing card_bin by transaction type:
type
DEPOSIT       0
WITHDRAWAL    2

Card deposits missing card_bin: 0


## Interim Read

The evidence summary is complete across the fields provided. The wallet file has two missing card BIN values, both on ACH withdrawal rows. I would not treat those as bad records by default because ACH withdrawals do not carry card BINs. A card deposit without a card BIN would be more concerning.

The data can support a first-pass analyst workflow and a feasibility memo. It is not enough for model training yet because the sample is small and does not include chargeback tables, KYC files, source-of-funds documents, or withdrawal destination details.
